# 📘 Notebook 13: Actor-Critic Methods

### Course: *From Random Moves to Winning Agents: Reinforcement Learning with Games*
**Instructor:** Ioannis Tsioulis

*Module D: Policy-Based Learning · Notebook 2 of 4 in this module · (13 of 18 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Define the **advantage** of an action and explain why it is a better weight than the return
- Train a **critic**, a network that estimates state values, alongside the policy (the **actor**)
- **Measure** the noise of a policy-gradient estimate and compare several estimators
- Explain the trade-off between **bias and variance**, and how **generalised advantage estimation (GAE)** moves between Monte Carlo and temporal-difference estimates
- Implement **A2C** with several environments running in parallel
- Judge the result honestly: what the critic buys, and what it does not

> **Prerequisites:** Notebook 12 (REINFORCE and the baseline), Notebook 05 (TD learning, the TD error, bootstrapping) and Notebook 03 (state values).
>
> 🔭 **Why this notebook matters later:** the data-collection function, the advantage estimator and the two networks written here are used unchanged by PPO in the next notebook. PPO is this notebook plus one idea.

> 📦 **Libraries used in this notebook.** **PyTorch**, **Gymnasium** (with Box2D for Lunar Lander) and **imageio**. No GPU is needed. The experiments on CartPole take seconds. Training the lander takes a few minutes.

In [ ]:
%pip install -q swig
%pip install -q "gymnasium[classic-control,box2d]>=1.3" imageio imageio-ffmpeg torch

In [ ]:
import base64
import time
import numpy as np
import matplotlib.pyplot as plt
import imageio
import gymnasium as gym
import torch
import torch.nn as nn
from IPython.display import HTML, display

torch.set_num_threads(2)


def show_video(frames, fps=30, width=420):
    """Play a list of pictures (RGB arrays) as a video inside the notebook."""
    imageio.mimsave("episode.mp4", frames, fps=fps, macro_block_size=1)
    data = base64.b64encode(open("episode.mp4", "rb").read()).decode()
    display(HTML(f'<video width="{width}" controls autoplay loop src="data:video/mp4;base64,{data}"></video>'))


def network(n_inputs, n_outputs, hidden=64):
    return nn.Sequential(nn.Linear(n_inputs, hidden), nn.Tanh(),
                         nn.Linear(hidden, hidden), nn.Tanh(),
                         nn.Linear(hidden, n_outputs))

## 1. Better than usual

### Intuition first
A football team wins 1 to 0. Was that a good result? Against the league leaders, a triumph. Against the bottom club, at home, a disappointment. A result means little until it is compared with what was to be **expected**.

REINFORCE weighted each action by the return that followed it. Notebook 12 ended with the observation that this number, taken alone, says little. A return of 50 is poor from a state where 200 is normal, and excellent from a state where the pole was already falling.

### Formal definition
The **advantage** of an action is how much better it is than the policy's average behaviour in that state:

$$A_\pi(s, a) = q_\pi(s, a) - v_\pi(s)$$

It is positive for actions that are better than usual and negative for actions that are worse. Using it as the weight in the policy gradient means using the **state value $v_\pi(s)$ as the baseline**. That baseline depends on the state and not on the action, so by Notebook 12, Section 6, the expected gradient is unchanged.

### Why it matters
The state value is the best simple baseline there is: it removes from the return everything that was due to the **situation** and leaves what was due to the **action**. The agent does not know $v_\pi$, of course. So it learns it, with a second network.

- The **actor** is the policy network $\pi(a \mid s; \boldsymbol\theta)$. It decides.
- The **critic** is a value network $\hat v(s; \mathbf{w})$. It judges.

The actor has no table of correct answers, only the critic's opinion of whether things went better or worse than expected. The critic in turn learns from what actually happens.

## 2. An actor, and a critic for it

To study the critic's effect in isolation we first need a policy to criticise. The cell below trains a small actor on CartPole with REINFORCE, as in Notebook 12, and stops as soon as its episodes last 60 steps on average, while it is still mediocre.

In [ ]:
env = gym.make("CartPole-v1")
GAMMA = 0.99


def returns_to_go(rewards, gamma=GAMMA):
    G, result = 0.0, []
    for reward in reversed(rewards):
        G = reward + gamma * G
        result.append(G)
    return torch.tensor(result[::-1], dtype=torch.float32)


def play_episode(actor, seed):
    """One episode with the given actor. Returns states, actions, rewards, and how it ended."""
    state, info = env.reset(seed=seed)
    states, actions, rewards = [], [], []
    terminated = truncated = False
    while not (terminated or truncated):
        states.append(state)
        with torch.no_grad():
            action = int(torch.distributions.Categorical(logits=actor(torch.as_tensor(state))).sample())
        state, reward, terminated, truncated, info = env.step(action)
        actions.append(action)
        rewards.append(reward)
    return (torch.as_tensor(np.array(states)), torch.as_tensor(actions), rewards, terminated, torch.as_tensor(state))


torch.manual_seed(0)
actor = network(4, 2)
optimiser = torch.optim.Adam(actor.parameters(), lr=3e-3)
recent = []
for episode in range(1000):                                  # a short REINFORCE training
    states, actions, rewards, terminated, last_state = play_episode(actor, seed=episode)
    G = returns_to_go(rewards)
    G = (G - G.mean()) / (G.std() + 1e-8)
    log_probabilities = torch.distributions.Categorical(logits=actor(states)).log_prob(actions)
    loss = -(log_probabilities * G).sum()
    optimiser.zero_grad()
    loss.backward()
    optimiser.step()
    recent.append(len(rewards))
    if np.mean(recent[-20:]) >= 60:                          # stop while the actor is still mediocre
        break

lengths = [len(play_episode(actor, seed=5000 + i)[2]) for i in range(30)]
print(f"training stopped after {episode + 1} episodes")
print(f"the actor keeps the pole up for {np.mean(lengths):.0f} steps on average (from {min(lengths)} to {max(lengths)})")

Now the critic. It has the state as input and **one** output, the estimated value of that state under this actor. Training it is ordinary regression: play episodes, compute the return that followed each state, and fit the network to predict it. This is the Monte Carlo prediction of Notebook 04 with a network in place of the table.

In [ ]:
torch.manual_seed(1)
critic = network(4, 1)
critic_optimiser = torch.optim.Adam(critic.parameters(), lr=1e-2)

episodes = [play_episode(actor, seed=10000 + i) for i in range(60)]
all_states = torch.cat([episode[0] for episode in episodes])
all_returns = torch.cat([returns_to_go(episode[2]) for episode in episodes])

for step in range(600):
    index = torch.randint(0, len(all_states), (256,))
    loss = (critic(all_states[index]).squeeze(1) - all_returns[index]).pow(2).mean()
    critic_optimiser.zero_grad()
    loss.backward()
    critic_optimiser.step()

with torch.no_grad():
    predicted = critic(all_states).squeeze(1)
print(f"average return in the data:       {all_returns.mean().item():6.1f}")
print(f"average error of the critic:      {(predicted - all_returns).abs().mean().item():6.1f}")
print(f"error of always guessing the mean: {(all_returns.mean() - all_returns).abs().mean().item():6.1f}")

The critic is far from exact: the return of a single episode is noisy, and no function of the state can predict it perfectly. But its error is clearly smaller than that of a constant guess, which means it has learned to tell good situations from bad ones.

## 3. Measuring the noise

Notebook 12 argued that a baseline reduces the variance of the gradient. With an actor and a critic in hand, we can **measure** it.

The experiment: keep the actor fixed. Collect 40 independent batches of 5 episodes. For each batch compute the policy gradient, a vector with one entry per parameter of the actor. If the estimator were noise-free, all 40 vectors would be identical. We report how much they scatter, relative to the length of their average: a **relative noise** of 1 means that the scatter is as large as the signal.

We compare five ways of weighting the log-probabilities:

| Name | Weight for the action at step $t$ |
|---|---|
| return | $G_t$, as in plain REINFORCE |
| return − mean | $G_t$ minus the average return of the episode |
| return − critic | $G_t - \hat v(S_t)$: the critic as baseline |
| TD error | $R_{t+1} + \gamma\,\hat v(S_{t+1}) - \hat v(S_t)$ |
| GAE | a mixture of the last two, explained in Section 4 |

In [ ]:
def gae_weights(rewards, values, next_values, lam, gamma=GAMMA):
    """Generalised advantage estimates for one episode (Section 4 explains the formula)."""
    deltas = torch.as_tensor(rewards) + gamma * next_values - values        # the TD errors
    weights, running = torch.zeros_like(deltas), 0.0
    for t in reversed(range(len(deltas))):
        running = deltas[t] + gamma * lam * running
        weights[t] = running
    return weights


def policy_gradient(batch, kind):
    """The gradient estimate from one batch of episodes, as one long vector."""
    loss, steps = 0.0, 0
    for states, actions, rewards, terminated, last_state in batch:
        G = returns_to_go(rewards)
        with torch.no_grad():
            values = critic(states).squeeze(1)
            last_value = torch.zeros(1) if terminated else critic(last_state)
        next_values = torch.cat([values[1:], last_value])
        if kind == "return":
            weights = G
        elif kind == "return - mean":
            weights = G - G.mean()
        elif kind == "return - critic":
            weights = G - values
        elif kind == "TD error":
            weights = gae_weights(rewards, values, next_values, lam=0.0)
        elif kind == "GAE (0.95)":
            weights = gae_weights(rewards, values, next_values, lam=0.95)
        log_probabilities = torch.distributions.Categorical(logits=actor(states)).log_prob(actions)
        loss = loss - (log_probabilities * weights).sum()
        steps += len(actions)
    actor.zero_grad()
    (loss / steps).backward()
    return torch.cat([parameter.grad.flatten() for parameter in actor.parameters()]).clone()


batches = [[play_episode(actor, seed=20000 + 100 * b + i) for i in range(5)] for b in range(40)]

reference = None
print(f"{'estimator':18s} {'relative noise':>15s} {'agreement with plain REINFORCE':>33s}")
for kind in ("return", "return - mean", "return - critic", "GAE (0.95)", "TD error"):
    gradients = torch.stack([policy_gradient(batch, kind) for batch in batches])
    mean = gradients.mean(dim=0)
    noise = (gradients.std(dim=0).pow(2).sum().sqrt() / mean.norm()).item()
    if reference is None:
        reference = mean
    agreement = (torch.dot(mean, reference) / (mean.norm() * reference.norm())).item()
    print(f"{kind:18s} {noise:15.2f} {agreement:33.2f}")

Read the first column of numbers from top to bottom.

- With the plain **return**, the scatter between batches is larger than the gradient itself. Each update is mostly noise.
- Subtracting the **mean** of the episode helps a little. A constant cannot tell good situations from bad ones.
- Using the **critic** as baseline lowers the noise further. This is the measurement behind the claim of Section 1.
- The **TD error** is the quietest of all, and **GAE** lies between it and the critic baseline.

The last column is the direction of the average gradient compared with plain REINFORCE, where 1 means "exactly the same direction". The estimators point largely the same way. They differ in how noisy they are, not in where they lead. (The smaller values in that column are partly noise themselves: 40 batches are few for pinning down the average of a quantity this noisy.)

## 4. Bootstrapping, bias and variance

### Intuition first
The weight "return − critic" still contains the whole return $G_t$: every random action and every random event until the end of the episode. Notebook 05 showed the alternative. Do not wait for the end. Take **one** real reward and then trust the estimate of what follows: $R_{t+1} + \gamma\,\hat v(S_{t+1})$. Subtracting the baseline $\hat v(S_t)$ gives the **TD error**

$$\delta_t = R_{t+1} + \gamma\,\hat v(S_{t+1}) - \hat v(S_t)$$

which is an estimate of the advantage of the action taken at step $t$. It depends on a single step of randomness, which is why it was the quietest row of the table.

There is a price. If the critic is wrong, the TD error is wrong in a systematic way, and no amount of averaging removes that. Statisticians call a systematic error **bias**.

| Estimate of the advantage | Uses | Variance | Bias |
|---|---|---|---|
| Monte Carlo: $G_t - \hat v(S_t)$ | all the real rewards to the end | high | none |
| TD: $\delta_t$ | one real reward, then the critic | low | as large as the critic's error |

### Formal definition
Between the two extremes lies a whole family. **Generalised advantage estimation (GAE)**, introduced by John Schulman and colleagues in 2016, adds up the TD errors of the following steps with weights that shrink by a factor $\gamma\lambda$ per step:

$$\hat A_t = \delta_t + (\gamma\lambda)\,\delta_{t+1} + (\gamma\lambda)^2\,\delta_{t+2} + \dots$$

The parameter $\lambda$ (lambda), between 0 and 1, is a dial:

- $\lambda = 0$ keeps only $\delta_t$: the TD estimate.
- $\lambda = 1$ makes the sum collapse to $G_t - \hat v(S_t)$: the Monte Carlo estimate.
- Values such as 0.95 trust the real rewards of the next few steps and the critic for the rest.

The sum is computed backwards with $\hat A_t = \delta_t + \gamma\lambda\,\hat A_{t+1}$, the same trick as for returns. That is what `gae_weights` above does.

In [ ]:
rewards = [1.0, 1.0, 1.0, 1.0]
values = torch.tensor([3.0, 2.5, 1.5, 1.0])               # what a critic might say about the four states
next_values = torch.tensor([2.5, 1.5, 1.0, 0.0])          # the episode ends after the fourth step

for lam in (0.0, 0.95, 1.0):
    print(f"lambda = {lam:<4}: advantages {gae_weights(rewards, values, next_values, lam, gamma=1.0).numpy().round(2)}")

print("Monte Carlo:    advantages", (returns_to_go(rewards, gamma=1.0) - values).numpy().round(2))

With $\lambda = 1$ the result equals the Monte Carlo advantages in the last line, and with $\lambda = 0$ each entry is one TD error.

## 5. A2C

### Formal definition
The **advantage actor-critic (A2C)** algorithm puts the pieces together, and adds two practical ideas.

- **Short rollouts.** The agent no longer waits for the end of an episode. It plays a fixed number of steps, estimates the advantages with GAE (using the critic for whatever lies beyond the last step), updates both networks, and continues from where it stopped.
- **Parallel environments.** Several copies of the game are played at the same time, each with its own random numbers. One update then uses steps from different episodes in different phases, which has the effect that the replay buffer had for DQN: the samples are less alike. It also makes good use of the processor, since the network is evaluated for all copies in one pass.

The loss has three parts:

$$\text{loss} = \underbrace{-\,\log\pi(A_t \mid S_t)\,\hat A_t}_{\text{actor}} \;+\; \tfrac12\,\underbrace{\big(\hat v(S_t) - \text{target}_t\big)^2}_{\text{critic}} \;-\; c\,\underbrace{H\big(\pi(\cdot \mid S_t)\big)}_{\text{entropy bonus}}$$

The critic's target is the advantage estimate plus the old value, which is the GAE version of a return. The **entropy** $H$ measures how spread out the action probabilities are. Rewarding it a little keeps the policy from becoming certain too early, which would end its exploration.

First, the function that plays `n_steps` in every environment and records everything. It is longer than it is deep: most lines store something.

In [ ]:
def collect(envs, states, actor, critic, n_steps, finished_returns, running_returns):
    """Play n_steps in every environment. Returns the recorded rollout and the new current states."""
    n_envs = len(envs)
    shape = (n_steps, n_envs)
    rollout = dict(states=np.zeros(shape + states.shape[1:], dtype=np.float32),
                   actions=np.zeros(shape, dtype=np.int64), rewards=np.zeros(shape, dtype=np.float32),
                   terminated=np.zeros(shape, dtype=np.float32), done=np.zeros(shape, dtype=np.float32),
                   values=np.zeros(shape, dtype=np.float32), next_values=np.zeros(shape, dtype=np.float32))
    for t in range(n_steps):
        with torch.no_grad():
            batch = torch.as_tensor(states)
            actions = torch.distributions.Categorical(logits=actor(batch)).sample().numpy()
            rollout["values"][t] = critic(batch).squeeze(1).numpy()
        rollout["states"][t], rollout["actions"][t] = states, actions
        next_states = np.zeros_like(states)
        for i, env in enumerate(envs):
            state, reward, terminated, truncated, info = env.step(int(actions[i]))
            rollout["rewards"][t, i] = reward
            rollout["terminated"][t, i] = terminated
            rollout["done"][t, i] = terminated or truncated
            next_states[i] = state                              # the state the action led to
            running_returns[i] += reward
            if terminated or truncated:
                finished_returns.append(running_returns[i])
                running_returns[i] = 0.0
                state, info = env.reset()
            states[i] = state                                   # where this environment continues
        with torch.no_grad():
            rollout["next_values"][t] = critic(torch.as_tensor(next_states)).squeeze(1).numpy()
    return rollout, states


def compute_gae(rollout, gamma, lam):
    """GAE for a rollout of shape (steps, environments), respecting the ends of episodes."""
    advantages = np.zeros_like(rollout["rewards"])
    running = np.zeros(advantages.shape[1])
    for t in reversed(range(len(advantages))):
        delta = (rollout["rewards"][t] + gamma * (1 - rollout["terminated"][t]) * rollout["next_values"][t]
                 - rollout["values"][t])
        running = delta + gamma * lam * (1 - rollout["done"][t]) * running
        advantages[t] = running
    return advantages

Two details repay attention, because they are the usual sources of bugs:

- `terminated` switches off the critic's estimate of the future, as in every notebook since 05. A time limit (`truncated`) does not: the state it led to still has a value.
- `done` stops the backward sum at the boundary between two episodes, so that the advantages of one episode are not mixed with the next.

In [ ]:
def a2c(env_name, total_steps, n_envs=8, n_steps=32, gamma=0.99, lam=0.95, learning_rate=1e-3,
        entropy_coef=0.01, seed=0, log=None):
    """Advantage actor-critic. Returns the actor, the critic and the return of every finished episode."""
    torch.manual_seed(seed)
    envs = [gym.make(env_name) for i in range(n_envs)]
    states = np.array([env.reset(seed=seed * 1000 + i)[0] for i, env in enumerate(envs)], dtype=np.float32)
    state_size, n_actions = states.shape[1], envs[0].action_space.n
    actor, critic = network(state_size, n_actions), network(state_size, 1)
    parameters = list(actor.parameters()) + list(critic.parameters())
    optimiser = torch.optim.Adam(parameters, lr=learning_rate)
    finished_returns, running_returns, steps, updates = [], np.zeros(n_envs), 0, 0

    while steps < total_steps:
        rollout, states = collect(envs, states, actor, critic, n_steps, finished_returns, running_returns)
        steps += n_steps * n_envs
        advantages = compute_gae(rollout, gamma, lam)
        targets = advantages + rollout["values"]

        s = torch.as_tensor(rollout["states"].reshape(-1, state_size))
        a = torch.as_tensor(rollout["actions"].reshape(-1))
        advantage = torch.as_tensor(advantages.reshape(-1))
        advantage = (advantage - advantage.mean()) / (advantage.std() + 1e-8)
        target = torch.as_tensor(targets.reshape(-1))

        distribution = torch.distributions.Categorical(logits=actor(s))
        actor_loss = -(distribution.log_prob(a) * advantage).mean()
        critic_loss = (critic(s).squeeze(1) - target).pow(2).mean()
        loss = actor_loss + 0.5 * critic_loss - entropy_coef * distribution.entropy().mean()
        optimiser.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(parameters, 0.5)
        optimiser.step()

        updates += 1
        if log and updates % log == 0 and finished_returns:
            print(f"step {steps:>7,}   episodes {len(finished_returns):>5}   "
                  f"average return of the last 50: {np.mean(finished_returns[-50:]):7.1f}")
    return actor, critic, np.array(finished_returns)

### CartPole

Three seeds, 100,000 steps each. With eight environments stepped together this takes only a few seconds per run.

In [ ]:
start = time.time()
runs = [a2c("CartPole-v1", 100000, seed=seed) for seed in range(3)]
print(f"three runs of 100,000 steps took {time.time() - start:.0f} seconds\n")


def smooth(values, window=30):
    return np.convolve(values, np.ones(window) / window, mode="valid")


plt.figure(figsize=(8, 4))
for seed, (actor_, critic_, returns) in enumerate(runs):
    plt.plot(smooth(returns), label=f"seed {seed}")
plt.axhline(500, color="gray", linestyle=":", label="perfect score")
plt.title("A2C on CartPole")
plt.xlabel("episode"); plt.ylabel("steps the pole stayed up (average of 30)"); plt.grid(True); plt.legend()
plt.show()

for seed, (actor_, critic_, returns) in enumerate(runs):
    print(f"seed {seed}: best 30-episode average {smooth(returns).max():4.0f},  last 30 episodes {returns[-30:].mean():4.0f}")

The agent learns, and it learns fast in terms of time on the clock. But look at the difference between the best and the last column, and between the seeds. The curves climb and then fall back. Less noise in the gradient has **not** produced a steady learner.

### Lunar Lander

The same function on the harder problem of Notebook 09. **This cell takes a few minutes**: two runs of 600,000 steps.

In [ ]:
start = time.time()
lander_runs = [a2c("LunarLander-v3", 600000, seed=seed, log=500) for seed in range(2)]
seconds = time.time() - start
print(f"\ntwo runs of 600,000 steps took {seconds:.0f} seconds")

plt.figure(figsize=(8, 4))
for seed, (actor_, critic_, returns) in enumerate(lander_runs):
    plt.plot(np.arange(49, len(returns)), smooth(returns, 50), label=f"seed {seed}")
plt.axhline(200, color="green", linestyle="--", label="solved")
plt.title("A2C on Lunar Lander")
plt.xlabel("episode"); plt.ylabel("return (average of 50)"); plt.grid(True); plt.legend()
plt.show()

for seed, (actor_, critic_, returns) in enumerate(lander_runs):
    print(f"seed {seed}: average return of the last 50 episodes {returns[-50:].mean():6.1f}")

In [ ]:
def record(env, actor, seed=0):
    state, info = env.reset(seed=seed)
    frames, total = [env.render()], 0.0
    terminated = truncated = False
    while not (terminated or truncated):
        with torch.no_grad():
            action = int(actor(torch.as_tensor(state, dtype=torch.float32)).argmax())     # the most probable action
        state, reward, terminated, truncated, info = env.step(action)
        total += reward
        frames.append(env.render())
    return frames, total


best = int(np.argmax([returns[-50:].mean() for actor_, critic_, returns in lander_runs]))
frames, total = record(gym.make("LunarLander-v3", render_mode="rgb_array"), lander_runs[best][0], seed=0)
print(f"the better of the two agents (seed {best}): return {total:.0f}")
show_video(frames[::2], fps=25)

## 6. An honest assessment

Both agents have moved a long way from random play, which scores far below zero, and they did so quickly. But the two seeds end in very different places, and neither run improves steadily: look for the stretches where the curve falls after having risen.

While preparing this notebook we ran the comparison more widely, always with 600,000 steps on Lunar Lander. The numbers are the average return over the last 50 episodes of each run.

| Agent | Three runs | Time per run on our machine |
|---|---|---|
| A2C (this notebook) | 180, −33, −110 | about 40 seconds |
| REINFORCE (Notebook 12), same number of steps | 164, 74 (two runs) | about 170 seconds |

So on this task the actor-critic was **not** more reliable than plain REINFORCE, and not better on average either. What it gained is of a different kind:

- **Speed.** Stepping eight environments together and updating every few steps made it about four times faster on the clock.
- **Learning during the episode.** It updates every 32 steps. REINFORCE must wait for an episode to end, which in a task without an end is never.
- **A quieter gradient**, as measured in Section 3.

Why does a quieter gradient not give a steadier learner? Because noise in the gradient is not the only danger. Every update changes the policy, the changed policy collects the next data, and that data produces the next update. One step that happens to be too large gives a worse policy, which collects worse data, from which it is hard to recover. Exercise 4 of Notebook 12 showed the same collapse with a large learning rate. Nothing in A2C limits how far a single update may move the policy.

That missing limit is the one idea PPO adds.

> ⚠️ **Common pitfalls**
>
> - **Letting the actor's loss change the critic's opinion.** The advantages are numbers computed beforehand (inside `torch.no_grad()` in `collect`). They must not be differentiated through.
> - **Mixing up `terminated` and `done`** in the advantage calculation. The two details explained under `compute_gae` are easy to get wrong and hard to notice.
> - **Evaluating by training reward alone.** During training the actor samples its actions. The video above uses the most probable action, which usually does a little better.
> - **Concluding from one seed.** With the first seed alone one would call A2C a success on the lander. With the second alone, a failure.

---
## ✏️ Exercises

The exercises use `network`, `gae_weights`, `returns_to_go`, `a2c`, `smooth`, `record`, and the `actor`, `critic` and `batches` of Sections 2 and 3.

### Exercise 1 (An advantage by hand)
In some state the critic says $\hat v(s) = 10$. The agent takes an action, receives a reward of 2, and lands in a state that the critic values at 9. With gamma = 0.9, compute the TD error. Was the action better or worse than expected? What would the TD error be if the episode had **ended** with that step?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
gamma = 0.9
td_error = 2 + gamma * 9 - 10
print(f"TD error: {td_error:+.2f}")
print(f"TD error if the episode ended: {2 + gamma * 0 - 10:+.2f}")
```

*2 + 0.9 × 9 = 10.1, so the action turned out 0.1 better than the critic expected, and its probability would be raised slightly. If the episode ended there, nothing follows the reward of 2, and the TD error is −8: much worse than expected, so the action would be strongly discouraged.*
</details>

### Exercise 2 (GAE at its two ends)
For the small example of Section 4 (`rewards`, `values`, `next_values`, gamma = 1), compute the four TD errors by hand. Then verify in Python that `gae_weights` with `lam=0` returns exactly those TD errors, and that with `lam=1` each entry is the sum of the TD errors from that step to the end.

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
rewards = [1.0, 1.0, 1.0, 1.0]
values = torch.tensor([3.0, 2.5, 1.5, 1.0])
next_values = torch.tensor([2.5, 1.5, 1.0, 0.0])

deltas = torch.tensor(rewards) + next_values - values
print("TD errors:            ", deltas.numpy())
print("GAE with lambda = 0:  ", gae_weights(rewards, values, next_values, 0.0, gamma=1.0).numpy())
print("sums from t to the end:", deltas.flip(0).cumsum(0).flip(0).numpy())
print("GAE with lambda = 1:  ", gae_weights(rewards, values, next_values, 1.0, gamma=1.0).numpy())
```

*When the TD errors are added up, each $\hat v(S_{t+1})$ cancels against the $-\hat v(S_{t+1})$ of the next term, and what remains is all the rewards minus $\hat v(S_t)$: the Monte Carlo advantage. The critic's errors cancel too, which is why $\lambda = 1$ has no bias.*
</details>

### Exercise 3 (A worse critic)
Section 3 measured the noise with a critic trained for 600 steps. Train a second critic for only 20 steps (copy the training cell, with a new network and optimiser), make `policy_gradient` use it by assigning it to the name `critic`, and print the relative noise of "return − critic" and "TD error" again, together with the agreement of each with the plain REINFORCE direction. Restore the original critic afterwards. Which estimator suffers more from a bad critic, and in which column?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
good_critic = critic

torch.manual_seed(2)
poor_critic = network(4, 1)
poor_optimiser = torch.optim.Adam(poor_critic.parameters(), lr=1e-2)
for step in range(20):
    index = torch.randint(0, len(all_states), (256,))
    loss = (poor_critic(all_states[index]).squeeze(1) - all_returns[index]).pow(2).mean()
    poor_optimiser.zero_grad()
    loss.backward()
    poor_optimiser.step()

plain = torch.stack([policy_gradient(batch, "return") for batch in batches]).mean(dim=0)
for name, which in (("good critic", good_critic), ("poor critic", poor_critic)):
    critic = which
    for kind in ("return - critic", "TD error"):
        gradients = torch.stack([policy_gradient(batch, kind) for batch in batches])
        mean = gradients.mean(dim=0)
        noise = (gradients.std(dim=0).pow(2).sum().sqrt() / mean.norm()).item()
        agreement = (torch.dot(mean, plain) / (mean.norm() * plain.norm())).item()
        print(f"{name}, {kind:16s} relative noise {noise:5.2f}   agreement with plain REINFORCE {agreement:5.2f}")

critic = good_critic
```

*A baseline can be arbitrarily wrong without changing the **direction** of the expected gradient: "return − critic" keeps its agreement with plain REINFORCE, and only loses some of its noise reduction. The TD error depends on the critic for its content, not only for its baseline. With a poor critic its average direction moves well away from the true one, and what is left of the signal is small compared with the scatter. That is the bias of Section 4, made visible.*
</details>

### Exercise 4 (The dial)
Train A2C on CartPole for 100,000 steps with `lam` = 0, 0.95 and 1, two seeds each, and print the average return over the whole training and over the last 30 episodes. Is there a clear winner? What does the result tell you about how much one can read into such a comparison?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
for lam in (0.0, 0.95, 1.0):
    for seed in range(2):
        actor_, critic_, returns = a2c("CartPole-v1", 100000, lam=lam, seed=seed)
        print(f"lambda = {lam:<4} seed {seed}: average over the whole training {returns.mean():4.0f},  "
              f"last 30 episodes {returns[-30:].mean():4.0f}")
```

*With $\lambda = 0$ the agent relies entirely on a critic that is poor at the start, and it learns slowest. Between 0.95 and 1 the difference is smaller than the difference between two seeds of the same setting. Values around 0.95 are the usual choice, but on a problem this small the experiment cannot tell them apart.*
</details>

### Exercise 5 (A swinging arm (a little harder))
`Acrobot-v1` is a two-link arm hanging down. The agent applies a torque at the middle joint (3 actions) and must swing the tip above a line. Every step costs −1 and an episode lasts at most 500 steps, so a random agent scores about −500. Train A2C for 150,000 steps with three seeds, print the average return of the last 50 episodes of each, and record a video of one trained agent. How does the reliability compare with CartPole and Lunar Lander?

In [ ]:
# Your solution here:


<details><summary>💡 Show solution</summary>

```python
acrobot_runs = [a2c("Acrobot-v1", 150000, seed=seed) for seed in range(3)]
for seed, (actor_, critic_, returns) in enumerate(acrobot_runs):
    print(f"seed {seed}: average return of the last 50 episodes {returns[-50:].mean():7.1f}")

frames, total = record(gym.make("Acrobot-v1", render_mode="rgb_array"), acrobot_runs[0][0], seed=0)
print(f"video: return {total:.0f}")
show_video(frames, fps=15)
```

*Here the three seeds agree well, and the agent learns to pump energy into the arm with a rhythmic torque. Acrobot has one clear way of getting better and no way of doing well and then ruining it, unlike a lander that can learn to hover and then forget how to land. How reliable an algorithm looks depends on the task as much as on the algorithm.*
</details>

## 🔑 Key takeaways

- The **advantage** $A(s, a) = q(s, a) - v(s)$ says whether an action was better or worse than usual in its state.
- An **actor-critic** agent has two networks: the actor chooses, the critic estimates state values and supplies the baseline.
- Measured directly, the critic as baseline clearly **reduced the noise** of the gradient, and TD-based estimates lowered it further.
- **GAE** mixes TD and Monte Carlo estimates with a dial $\lambda$: lower variance at one end, no bias at the other.
- **A2C** learns from short rollouts of several parallel environments, so it is fast and does not need episodes to end.
- A quieter gradient alone did **not** make learning reliable: nothing stops one update from moving the policy too far.

---
**Next:** *Notebook 14: Proximal Policy Optimisation*, where one more idea makes this agent reliable.

---
*© Ioannis Tsioulis. *From Random Moves to Winning Agents: Reinforcement Learning with Games*. Prepared for educational use.*